# 18 · Convergence of P1D and of the wedges of $P$ with the number of skewers (TNG50)

The skewers are read from the **skewer store** (`docs/skewer_store.md`), filled by
`scripts/production/extract_skewers.py`: nested $N \times N$ lattices of skewers ($x_i = i\,L/N$, $N = 2, 4, \dots$)
through one snapshot along $x$, $y$ and $z$, with notebook 11's $\tau$ (snapshot frame, linear Hubble flow and
detuning, Harris' Voigt, periodic over one box). Every lattice goes through the same call as notebook 17
(`summary_stats.flux_statistics`: transmission, overflux, P1D, the 3D statistics on the lattice), on the bins of the
largest lattice. Each statistic is drawn for every lattice, with its residual against the largest lattice
$N_{\rm SK}$ in a row below, $100\,(X_N / X_{N_{\rm SK}} - 1)$ in percent, so you can read off the percentage
to which each lattice has converged.

- **Part A — the store:** the dataset and the lattices complete on every axis.
- **Part B — the bins and the statistics of every lattice** (notebook 10's and 11's bins, with the TNG50 lattice).
- **Part C — P1D** of every line and of the total, and its residuals.
- **Part D — the wedges of $P(k, \mu)$** of the total, one panel per wedge (notebook 10's `panels=True`), each with its
  residuals.
- **Part E — summary:** the percentage levels per lattice.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()                                    # interactive figures in a live notebook
use_style()
from skewerjax.utils.plotting import add_panel_letters, field_names, line_names
from skewerjax.core import LineBundle
from skewerjax.simulations import format_datasets, list_datasets, load_skewers, select_datasets

## Part A — Inputs and the store

- `STORE`: the `--out` of the script (`$SKEWERJAX_STORE`, default `/home/STORAGE/SKEWERS_SkewerJAX`).
- `SIM`, `SNAP`, `HASH`: the dataset (`HASH = None`: the configuration with the largest lattice complete on every
  axis of `AXES`).
- `AXES`: the lines of sight of the ensemble (1, 2, 3 = $x$, $y$, $z$).
- `N_MIN`, `N_MAX`: the lattices compared, every power of two from `N_MIN` to `N_MAX` complete on every axis.
  `N_MAX = None` takes the largest stored. **The reference is `N_MAX`**: a quick check runs with `N_MAX = 16`
  (seconds), the full series with `N_MAX = 256` (about 5 minutes and 40 GB of memory at peak; Part B gives the
  time of each step).

The lattices are nested, so loading the $N \times N$ lattice reads the skewers of every coarser level as well. That
reuse is free, because each lattice's skewers are a subset of the next one's (notebook 17).

In [ ]:
STORE = Path(os.environ.get("SKEWERJAX_STORE", "/home/STORAGE/SKEWERS_SkewerJAX"))   # the --out of extract_skewers.py
SIM, SNAP = "TNG50-2", 25
HASH = None                                         # e.g. "cd782fdb19"; None: the configuration with the largest lattice
AXES = (1, 2, 3)                                    # the lines of sight x, y, z: the ensemble
N_MIN, N_MAX = 2, 128                              # the lattices compared; N_MAX the reference (None: the largest stored; 16: a quick check)
ROWS = list_datasets(STORE)
if not ROWS:
    raise FileNotFoundError(f"no dataset in the store {STORE}: set STORE (or $SKEWERJAX_STORE) to the --out of extract_skewers.py")
print(f"store {STORE}:\n{format_datasets(ROWS)}")
SEL = select_datasets(STORE, SIM, SNAP, HASH, axes=AXES)
HASH, AXES = SEL["hash"], SEL["axes"]
N_STORED = SEL["n_values"]                          # the lattices complete on every axis
N_MAX = N_STORED[-1] if N_MAX is None else N_MAX
if N_MAX not in N_STORED:
    raise ValueError(f"N_MAX = {N_MAX} is not complete on the axes {AXES}: the complete lattices are {N_STORED}")
N_VALUES = [n for n in N_STORED if N_MIN <= n <= N_MAX]   # the lattices compared, increasing
if len(N_VALUES) < 2:
    raise ValueError(f"at least two lattices are needed between N_MIN = {N_MIN} and N_MAX = {N_MAX} (complete: {N_STORED})")
N_SK = N_MAX                                        # the reference
S0 = load_skewers(STORE, SIM, SNAP, AXES[0], HASH, 2)   # the metadata (the smallest lattice: quick)
CFG = S0.config
LINE_IDS = S0.line_ids
LINES = LineBundle(list(LINE_IDS), ref_line_id=LINE_IDS[0])
L, u, V, DV, PERIOD = S0.L, S0.u, S0.v, S0.dv, S0.period   # [cMpc], aH [km/s per cMpc], the pixel centres, dv, v_per [km/s]
N_PIX = V.size
Z, A_SNAP = CFG["snapshot"]["redshift"], CFG["snapshot"]["a"]
MEAN = "redshift"                                   # T_bar(z), as notebooks 10 and 11
MU_EDGES_XI = (0.0, 0.5, 0.8, 0.95, 1.0)            # notebook 10's and 11's wedges of xi (needed by the grid)
MU_EDGES_P = np.sin(np.linspace(0.0, np.pi / 2, 17))   # notebook 10's and 11's 16 wedges of P, evenly spaced in angle
NAMES_L = line_names(LINE_IDS)
o = CFG["options"]
print(f"{SIM} snapshot {SNAP}, z = {Z:.4f}: dataset {HASH} (preset {CFG.get('preset')!r}) on the axes {AXES}; stored N = {N_STORED}; "
      f"compared N = {N_VALUES}, the reference N_SK = {N_SK}")
print(f"{N_PIX} pixels of {DV:.4f} km/s over v_per = {PERIOD:.2f} km/s; u = {u:.3f} km/s per cMpc; L = {L:.3f} cMpc")
print("tau: " + ", ".join(f"{k}={v}" for k, v in o["tau"].items()) + f"; pieces {o['pieces']['pieces']}; {o['precision']}")

## Part B — The bins and the statistics of every lattice

The bins are those of notebooks 10 and 11, adapted to the TNG50 lattice. In 3D they are set by the largest lattice
(the transverse spacing $L/N_{\rm SK}$ and the pixel $dv/u$, $|k|$ one bin per mode then steps of $0.4\,k_f$ up to the
smaller Nyquist frequency), and `lattice_grid` cuts them at each lattice's own Nyquist frequency $\pi N/L$. P1D is
not rebinned: it stays on its native modes $k_m = 2\pi m/v_{\rm per}$, $m = 1, \dots, N_{\rm pix}/2$ (the same for every
lattice: the same pixels). Each lattice is read from the store (its
skewers are a subset of the largest lattice's, stride $N_{\rm SK}/N$) and goes through `flux_statistics`.
`lattice_convergence` then takes the ratios $X_N / X_{N_{\rm SK}}$ (for P1D of every line and of the total; for the
wedges, of every field in `WEDGE_FIELDS`). For Part C, P1D of the total overflux is also split order by order
($|A| + |B|$, `decompose_1d` on each line of sight with the lattice's $\bar T$, then `ensemble_1d`), as notebooks 10
and 11 (D5).

**Progress and time.** The bar counts every step: per lattice, the reading, the bins (`lattice_grid`), the
transmissions, then per line of sight P1D with $\xi_{\rm 1D}$ and the 3D statistics of each field, and the order decomposition. Its text names the
step under way, and one line per lattice gives the time of each step. The cost scales with the number of lattice
points $N^2 N_{\rm pix}$, about 4 times per doubling. Measured at $N = 256$ (2.6e8 points, 3,961 pixels): the bins
45 s, each 3D statistic 7 s (9 of them), P1D with $\xi_{\rm 1D}$ 26 s per line of sight. That is about 3 minutes, and
the series 2 to 256 about 5 minutes (the order decomposition adds about 45 s at $N = 256$); the peak memory is about 40 GB. At `N_MAX = 16` it all takes seconds.

In [ ]:
import re
from tqdm.auto import tqdm
from skewerjax.summary_stats import bao_maxima, decompose_1d, ensemble_1d, flux_statistics, flux_statistics_steps, lattice_convergence, lattice_grid, set_k_edges
from skewerjax.utils.reference_data import P1D_DESI_DR1, nearest_reference
from skewerjax.core import sound_horizon_drag
from skewerjax.simulations import TNG_COSMOLOGY as TNG
D_PERP = L / N_SK
dv_c = DV / u                                                   # a pixel [cMpc]
dr_c = max(D_PERP, dv_c)
RPERP_EDGES = np.arange(0, L / 2 + D_PERP, D_PERP)              # r_perp: width L / N_SK to L / 2
R_EDGES = np.arange(0, L / 2 + dr_c, dr_c)                      # |r|: the coarser spacing, to L / 2
k_nyq = min(np.pi / D_PERP, np.pi / dv_c)
K_EDGES = set_k_edges(2 * np.pi / L, k_1=3.5 * 2 * np.pi / L, k_2=k_nyq, k_max=k_nyq, dk_factor=0.4)   # |k|: one bin per mode, then steps of 0.4 k_f
t0 = time.perf_counter()
GRID = lattice_grid(N_SK, N_SK, N_PIX, L, D_PERP, k_edges=K_EDGES, r_edges=R_EDGES, rperp_edges=RPERP_EDGES,
                    xi_mu_edges=MU_EDGES_XI, pk_mu_edges=MU_EDGES_P, log=print)
EDGES = dict(k_edges=GRID.k_edges, r_edges=GRID.r_edges, rperp_edges=GRID.rperp_edges)   # cut at the full coverage of N_SK
print(f"the bins of N_SK = {N_SK} in {time.perf_counter() - t0:.1f} s")
K_LIMITS_1D = (2 * np.pi / PERIOD, np.pi / DV)
P1D_REF = nearest_reference(P1D_DESI_DR1, Z)
P1D_K_MAX = float(np.max(P1D_REF["k"]))                          # [s/km]: the range of DESI DR1 at the snapshot's z (the summary)
R_BAO = float(sound_horizon_drag(TNG["Omega_b"] * CFG["snapshot"]["h"]**2, CFG["snapshot"]["omega_m"] * CFG["snapshot"]["h"]**2))
BAO_K = bao_maxima(1.0, r_bao=R_BAO)                             # the maxima of j_0(k r_d) [cMpc^-1], as notebook 11
A_PIX = np.full(N_PIX, A_SNAP)
WEDGE_FIELDS = ("total",)                                        # the fields of the wedges: add LINE_IDS for every line
STATS, ORDERS, TIMES = {}, {}, {}
per_lattice = 3 + flux_statistics_steps(len(AXES), len(LINE_IDS))        # reading, bins, the steps of flux_statistics, the orders
bar = tqdm(total=len(N_VALUES) * per_lattice, desc="lattices", unit="step")
t0 = time.perf_counter()
for n in N_VALUES:
    steps, t1 = [], time.perf_counter()

    def log(msg, n=n, steps=steps):                                     # one step of this lattice: the bar and its time
        steps.append(msg)
        bar.set_postfix_str(f"N = {n}: {msg}")
        bar.update(1)

    bar.set_postfix_str(f"N = {n}: reading {len(AXES)} x {n * n:,} skewers")
    tt = time.perf_counter()
    tau_n = np.stack([load_skewers(STORE, SIM, SNAP, a, HASH, n).tau for a in AXES])   # (n_ax, N_lines, n^2, N_PIX)
    log(f"read {tau_n.nbytes / 1e9:.2f} GB of tau ({time.perf_counter() - tt:.1f} s)")
    tt = time.perf_counter()
    grid_n = GRID if n == N_SK else lattice_grid(n, n, N_PIX, L, L / n, **EDGES, xi_mu_edges=MU_EDGES_XI, pk_mu_edges=MU_EDGES_P)
    log(f"bins of the {n} x {N_PIX} x {n} lattice ({time.perf_counter() - tt:.1f} s)")
    STATS[n] = flux_statistics(tau_n, DV, grid_n, line_ids=LINE_IDS, mean=MEAN, a_pix=A_PIX, log=log)
    tt = time.perf_counter()
    ORDERS[n] = ensemble_1d([decompose_1d(tau_n[i], DV, STATS[n]["T_bar"][i], mean=MEAN) for i in range(len(AXES))])   # P1D by order (notebook 11, D5)
    log(f"P1D by order ({time.perf_counter() - tt:.1f} s)")
    del tau_n, grid_n
    TIMES[n] = time.perf_counter() - t1
    secs = lambda key: sum(float(re.search(r"\(([\d.]+) s\)$", m).group(1)) for m in steps if key in m)
    tqdm.write(f"N = {n:4d} ({len(AXES)} x {n * n:,} skewers): {TIMES[n]:6.1f} s = reading {secs('read'):.1f} + bins {secs('bins'):.1f} "
               f"+ transmissions {secs('transmissions'):.1f} + P1D and xi1D {secs('P1D'):.1f} + 3D statistics {secs('3D'):.1f} + orders {secs('order'):.1f} s")
bar.close()
FIELDS = STATS[N_SK]["fields"]
NAMES = field_names(FIELDS)
CONV = lattice_convergence(STATS, p1d_k_max=P1D_K_MAX, fields_3d=WEDGE_FIELDS)   # P1D on its native modes (no k_edges_1d)
print(f"every lattice in {time.perf_counter() - t0:.0f} s; P1D on {CONV['p1d']['k'].size} native modes of k; P(k, mu) in {MU_EDGES_P.size - 1} wedges x "
      f"{CONV['P_mu']['total']['ref'].shape[-1]} bins of |k| up to {GRID.k_edges[-1]:.3f} cMpc^-1")

## Part C — P1D order by order

P1D of the total overflux split by order $|A| + |B|$ (notebook 11, D5: `plot_p1d_orders`), one panel per order and a
last one for the total (their sum), with every lattice on each. Orders 0 and 1 are not drawn: order 0 lives at $k = 0$ only, and order 1 vanishes because
$\langle\delta_A\rangle = 0$. P1D is drawn on its native modes $k_m = 2\pi m/v_{\rm per}$, $m = 1, \dots, N_{\rm pix}/2$,
one point per mode (no rebinning, as notebooks 10 and 11), and the grey bands lie outside $[2\pi/v_{\rm per}, \pi/dv]$.
With `SHOW_DESI`, the total panel also shows the DESI DR1 P1D of Chaves-Montero et al. at every redshift (one colour
per $z$, as notebook 11, D4; the snapshot is at the $z$ in the title). Above, the largest lattice is thicker, with the error of the mean over the lines of sight
as error bars. Below, the residual $100\,(P_N - P_{N_{\rm SK}})/|P_{N_{\rm SK}}|$ [%] on a symlog axis, linear within
$\pm 1\%$, with dotted lines at $\pm 1\%$.

The options of `plot_p1d_lattices`: `quantity` "P" or "kP_pi" ($kP/\pi$); `k_scale` "log" or "linear"; `y_scale`
"symlog" (the cross orders change sign), "linear" or "log"; `residual` "relative" or "max" (divided by the largest
$|P_{N_{\rm SK}}|$ of the panel, which avoids the spikes of a cross order near its zeros); `residual_scale` "symlog" or "linear"
(the residual rows; `residual_limit` sets their range in %); `total` (default True) the panel of the total; `reference` a measured P1D on the total panel; `k_edges` opt-in display bins; `orders`, `ncols`.


In [ ]:
from skewerjax.utils.plotting import plot_p1d_lattices
NSK = r"$N_{\rm SK}$"
SHOW_DESI = True                                                 # the DESI DR1 P1D (every z) on the total panel
fig, axs, RES_P1D, K_P1D = plot_p1d_lattices(
    ORDERS, k_limits=K_LIMITS_1D, quantity="kP_pi", k_scale="linear", y_scale="symlog", residual="relative", residual_scale="symlog", total=True,
    reference=P1D_DESI_DR1 if SHOW_DESI else None,
    kms_per_cMpc=u, title=f"$P_{{\\rm 1D}}$ of the total by order, every lattice against {NSK} = {N_SK} ({SIM}, z = {Z:.2f}, {len(AXES)} lines of sight)")
add_panel_letters(axs[:, 0])
plt.show()
for o, d in RES_P1D.items():
    print(f"{'order ' + str(o) if o != 'total' else 'total'}: |residual| over the {K_P1D.size} native modes, median / max [%]: " + "; ".join(
          f"N = {n}: {np.nanmedian(np.abs(r)):.2g} / {np.nanmax(np.abs(r)):.2g}" for n, r in d.items()))

**Reading.** Each skewer samples the whole line-of-sight power, so the mean over $N^2$ skewers per axis differs from
the largest lattice's by its sample variance. That variance falls as $1/N^2$ while the skewers are farther apart than
the transverse correlation length, so the residuals shrink by about 4 at each doubling until they reach the error bars of the reference.
Order 2 holds the auto-powers (Ly$\alpha$ dominates it); order 3, the Ly$\alpha$–Si III cross terms, oscillates with
$\cos(k\,\Delta v)$ and changes sign, so its relative residual spikes at its zeros (`residual="max"` avoids them);
order 4 is the Si III auto-power and its products. At high $k$, and for the weak Si III terms, the power of a few skewers is dominated by their rare strong
absorbers, so a coarse lattice converges there last. The residuals are measured against the largest lattice, not
the truth: the lattice $N_{\rm SK}/2$ is a quarter of the skewers of $N_{\rm SK}$ (shared), so its residual is only a
rough bound on the error of $N_{\rm SK}$ itself.

## Part D — The wedges of $P(k, \mu)$

One panel per wedge of `MU_EDGES_P` (16, evenly spaced in angle to the line of sight; notebook 10's `panels=True`).
Above, $P(k, \mu)$ of the total for every lattice at the bin centres of the largest lattice; a lattice of $N$ skewers
per axis has no transverse mode beyond $\pi N/L$, so its curve stops there. Below, the residual
$100\,(P_N/P_{N_{\rm SK}} - 1)$ [%], with the same symlog axis and bands as Part C. The grey dotted lines are the BAO
maxima of $j_0(k\,r_d)$.

In [ ]:
from skewerjax.utils.plotting import plot_pk_mu_wedges_lattices
RES_PMU = {}
for f in WEDGE_FIELDS:
    fig, axs, RES_PMU[f] = plot_pk_mu_wedges_lattices(CONV, field=f, mu_edges=MU_EDGES_P, ncols=4, bao_k=None,
                                                      title=f"$P(k, \\mu)$ of {NAMES[f]} for every lattice against {NSK} = {N_SK}")
    add_panel_letters(axs[:, 0])
    plt.show()
    c = CONV["P_mu"][f]
    print(f"{NAMES[f]}: |residual| over every bin each lattice covers, median / max [%], per wedge group (mu < 0.5, 0.5-0.9, > 0.9):")
    mu_mid = 0.5 * (MU_EDGES_P[1:] + MU_EDGES_P[:-1])
    groups = {"mu < 0.5": mu_mid < 0.5, "0.5 < mu < 0.9": (mu_mid >= 0.5) & (mu_mid < 0.9), "mu > 0.9": mu_mid >= 0.9}
    for n in N_VALUES[:-1]:
        parts = []
        for g, sel in groups.items():
            r = np.abs(np.concatenate([RES_PMU[f][w][n] for w in np.flatnonzero(sel)]))
            rf = r[np.isfinite(r)]
            parts.append(f"{g}: {np.median(rf):.2g} / {np.max(rf):.2g}" if rf.size else f"{g}: -")
        print(f"  N = {n:4d}: " + "; ".join(parts))

**Reading.** In each wedge the coarse lattices stop at their Nyquist frequency $\pi N/L$, which is reached first
across the line of sight. Below it, the transverse power aliased from the scales between the skewers pushes the ratio
above 1, most at low $\mu$ and least along the line of sight, where the pixels resolve every scale. The near-$\mu = 1$
wedges converge first; the low-$\mu$ wedges need the finest lattices.

## Part E — Summary: the percentage levels per lattice

For each lattice, the median and the largest $|X_N/X_{N_{\rm SK}} - 1|$ [%] over the modes or bins it covers: P1D of the total
on its native modes within the DESI range, and $P(k, \mu)$ of the total over every wedge. The last column is the median relative error of
the largest lattice's estimate: residuals below it cannot be told apart from the reference's own noise.

In [ ]:
j_tot = list(FIELDS).index("total")
kin = CONV["p1d"]["k"] <= P1D_K_MAX                                  # the DESI DR1 range at the snapshot's z
r1 = {n: 100 * np.abs(CONV["p1d"]["ratio"][n][j_tot] - 1)[kin] for n in N_VALUES[:-1]}
r3 = {n: np.abs(np.concatenate([RES_PMU["total"][w][n] for w in RES_PMU["total"]])) for n in N_VALUES[:-1]}
e1 = 100 * CONV["summary"]["P1D total"]["ref_err"]
e3 = 100 * CONV["summary"]["P(k, mu) total"]["ref_err"]
pct = lambda r, f: f"{f(r[np.isfinite(r)]):.3g}" if np.isfinite(r).any() else "-"   # "-": no bin (a lattice below the first |k| bin)
print(f"{'N':>5} {'skewers/axis':>13} | {'P1D total: median':>18} {'max':>8} | {'P(k,mu) total: median':>22} {'max':>8}   [%]")
for n in N_VALUES[:-1]:
    print(f"{n:5d} {n * n:13,d} | {pct(r1[n], np.median):>18} {pct(r1[n], np.max):>8} | {pct(r3[n], np.median):>22} {pct(r3[n], np.max):>8}")
print(f"the median relative error of N_SK = {N_SK}: P1D total {e1:.3g} %, P(k, mu) total {e3:.3g} %")